In [ ]:
import sys

import pandas as pd
import pickle
from trust_library.core import TrustEvaluator
from trust_library.factsheet import Factsheet

import matplotlib.pyplot as plt
import seaborn as sns
import json

from trust_library.utils import to_json_safe
import plotly.express as px

import numpy as np
from sklearn.model_selection import train_test_split

from UniversarSklearnWrapper import UniversalSklearnWrapper

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, callbacks

from sklearn.linear_model import LogisticRegression

from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.datasets import fetch_kddcup99
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

Random Forest Classifier

In [2]:
from sklearn.ensemble import RandomForestClassifier
import pickle

#los datos train y test están en train.csv y test.csv, los cargamos

import pandas as pd
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
X_train = train.drop(columns=["Target"])
y_train = train["Target"]

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

with open("model.pkl", "wb") as f:
    pickle.dump(clf, f)


SVM

In [ ]:
from sklearn.svm import SVC
import pickle
import pandas as pd

# cargar datasets
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

X_train = train.drop(columns=["Target"])
y_train = train["Target"]

# modelo SVM
clf = SVC(
    kernel="rbf",
    probability=True,   # necesario si luego usas predict_proba
    random_state=42
)

clf.fit(X_train, y_train)

# guardar modelo
with open("model_SVM.pkl", "wb") as f:
    pickle.dump(clf, f)

Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier
import pickle

#los datos train y test están en train.csv y test.csv, los cargamos

import pandas as pd
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
X_train = train.drop(columns=["Target"])
y_train = train["Target"]

clf = DecisionTreeClassifier(random_state=42, max_depth=5)
clf.fit(X_train, y_train)

with open("model_tree.pkl", "wb") as f:
    pickle.dump(clf, f)


Red neuronal

In [ ]:
# 1. URL DE DATOS REALES (Heart Disease Dataset)
url_data = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"]
df = pd.read_csv(url_data, names=columns, na_values="?")
df = df.dropna()

# Binarizar el target (0 = sin enfermedad, 1 = con enfermedad)
df["target"] = (df["target"] > 0).astype(int)

X = df.drop("target", axis=1).values
y = df["target"].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. DEFINIR LA RED (Real y funcional)
class HeartDiseaseNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )
    def forward(self, x):
        return self.net(x)

# 3. ENTRENAR Y GUARDAR (Simulando lo que harías en producción)
pytorch_model = HeartDiseaseNet(input_dim=X_train.shape[1])
optimizer = torch.optim.Adam(pytorch_model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

# ESCALAMOS LOS DATOS (Vital para Redes Neuronales)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

for epoch in range(100): # Entrenamiento flash
    optimizer.zero_grad()
    loss = criterion(pytorch_model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

# Guardamos el modelo "real"
torch.save(pytorch_model.state_dict(), "models_and_data/modelo_corazon_real.pth")
print("Modelo guardado como 'models_and_data/modelo_corazon_real.pth'")

# ==========================================
# 4. USO DEL WRAPPER DEFINITIVO CON TU LIBRERÍA
# ==========================================
# Cargas el modelo desde el archivo
modelo_cargado = HeartDiseaseNet(input_dim=X_train.shape[1])
modelo_cargado.load_state_dict(torch.load("models_and_data/modelo_corazon_real.pth"))

# Aplicas el Wrapper Definitivo
wrapped_model = UniversalSklearnWrapper(model=modelo_cargado)

Test Accuracy: 0.8300


Regresión

Cargando dataset de Diabetes...
Archivos 'train_regression.csv' y 'test_regression.csv' guardados exitosamente.
Entrenando el modelo de Regresión Lineal...
Modelo entrenado. MSE: 2900.19 | R2 Score: 0.45



Multiclase

Cargando dataset Wine...
Archivos 'train_multiclass.csv' y 'test_multiclass.csv' guardados exitosamente.
Entrenando el modelo RandomForestClassifier...
Modelo entrenado. Accuracy: 1.00
Reporte de clasificación:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      1.00      1.00        14
           2       1.00      1.00      1.00        10

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36

